In [10]:
# =============================================================================
# DIRECT PREFERENCE OPTIMIZATION (DPO) — preference tuning for Correction-GPT
# =============================================================================
#
# ---------------------------------------------------------------------------
# WHERE ARE WE ON THE PATH? (picture the 3 stages again)
# ---------------------------------------------------------------------------
#   1) PRETRAIN   — read tons of text → learn language (next-token guessing)
#   2) SFT        — flashcards: one ideal answer per prompt (notebook 9)
#   3) PREFERENCE — THIS NOTEBOOK
#        For the SAME prompt, humans (or you) say:
#          "answer A is better than answer B"
#        Model learns to prefer A over B — tone, brevity, confidence, etc.
#
# Easy analogy:
#   SFT  = show the student the answer key once.
#   DPO  = show two essays and circle the better one.
#         "Write more like THIS, less like THAT."
#
# Why not stop at SFT?
#   SFT only shows ONE gold reply. It does NOT teach:
#     - short > rambling
#     - confident correction > "I'm not sure, maybe ask someone..."
#     - whisper style > essay style
#   Preferences fill that gap WITHOUT needing a separate reward model
#   + RL loop (classic RLHF). DPO does preference learning in one loss.
#
#
# ---------------------------------------------------------------------------
# TOPIC: what is DPO? (deep but easy)
# ---------------------------------------------------------------------------
# Classic RLHF (rough cartoon):
#
#   SFT model ──▶ sample answers ──▶ humans rank them
#                      │
#                      ▼
#              train a REWARD model (scores "how good")
#                      │
#                      ▼
#              RL (PPO) nudges policy to get high reward
#              (tricky: unstable, many knobs)
#
# DPO shortcut:
#
#   Keep a frozen REFERENCE model (usually the SFT checkpoint).
#   Train POLICY model so that:
#     preferred (chosen) answer becomes MORE likely than rejected,
#     relative to what the reference already thought.
#
# Sticky intuition (no formulas yet):
#   ┌──────── prompt ────────┐
#   │ Ground truth + user    │
#   └───────────┬────────────┘
#               │
#        ┌──────┴──────┐
#        ▼             ▼
#   CHOSEN ✅      REJECTED ❌
#   short, clear   long, hedgy, unsure
#        │             │
#        └──────┬──────┘
#               ▼
#   "Push probability UP on chosen tokens,
#    DOWN on rejected tokens — but don't
#    wander too far from the SFT reference."
#
# Same Correction-GPT product idea:
#   Earpiece should whisper a CLEAN correction, not a nervous paragraph.
#
#
# ---------------------------------------------------------------------------
# THIS CELL — build a tiny PREFERENCE dataset + train BPE on it
# ---------------------------------------------------------------------------
# Each row has THREE strings:
#   prompt   — the situation (truth + what user said)
#   chosen   — the answer we WANT (concise Correction: ...)
#   rejected — the answer we DON'T WANT (verbose / uncertain)
#
# Later cells will score both completions under the policy + reference
# and apply the DPO loss. Today we only prepare data + tokenizer.
#

from pathlib import Path
import sys
import json

# week2/ on path so `from tokenizer import BPETokenizer` works
_WEEK2 = Path.cwd() / "week2"
if _WEEK2.is_dir():
    sys.path.insert(0, str(_WEEK2.resolve()))
elif Path.cwd().name == "week2":
    sys.path.insert(0, str(Path.cwd().resolve()))
else:
    sys.path.insert(0, str(Path("week2").resolve()))

import importlib
import tokenizer as _tokenizer_mod
importlib.reload(_tokenizer_mod)  # pick up save/load if kernel had an old import
from tokenizer import BPETokenizer  # week2/tokenizer.py (notebook 8)
import torch

# ---------------------------------------------------------------------------
# Preference Dataset (Chosen vs Rejected)
# ---------------------------------------------------------------------------
# Picture for ONE row:
#
#   PROMPT
#   ┌─────────────────────────────────────────────────────────┐
#   │ Ground truth: The price is $500/month                   │
#   │ User said:    It costs $300 a month                     │
#   └─────────────────────────────────────────────────────────┘
#            │                              │
#            ▼ chosen ✅                    ▼ rejected ❌
#   ┌──────────────────────────┐   ┌────────────────────────────────────┐
#   │ Correction: The price is │   │ Well, you may want to double-check │
#   │ $500 per month, not $300.│   │ ... I'm not entirely sure...       │
#   └──────────────────────────┘   └────────────────────────────────────┘
#
# Sticky: SAME prompt, TWO futures. DPO learns which future to prefer.

preference_data = [
    {
        "prompt": "Ground truth: The price is $500/month\nUser said: It costs $300 a month",
        "chosen": "Correction: The price is $500 per month, not $300.",
        "rejected": (
            "Well, you may want to double-check that because I think the actual "
            "price is somewhere around $500, but I'm not entirely sure, let me look it up..."
        ),
    },
    {
        "prompt": "Ground truth: We are SOC 2 compliant\nUser said: Are we SOC 2?",
        "chosen": "Correction: Yes, we are SOC 2 compliant.",
        "rejected": (
            "I believe we are SOC 2 compliant, but I'm not 100% certain, "
            "maybe ask the security team later."
        ),
    },
    {
        "prompt": "Ground truth: The API rate limit is 1000/min\nUser said: The limit is 500 per minute",
        "chosen": "Correction: The rate limit is 1000 requests per minute, not 500.",
        "rejected": (
            "Actually the rate limit is higher, it's around a thousand, but don't "
            "quote me on that, please refer to the docs."
        ),
    },
    # Later: add more pairs for tone, length, confidence, etc.
]

print(f"preference pairs: {len(preference_data)}")
print("Sample pair:\n", json.dumps(preference_data[0], indent=2))


# ---------------------------------------------------------------------------
# Format prompt the SAME way as SFT (Alpaca headings)
# ---------------------------------------------------------------------------
# DPO still conditions on an instruction string. We only change what we
# OPTIMIZE (chosen vs rejected), not the heading style.

def format_prompt(ground_truth, user_utterance):
    """Alpaca-style prompt ending at ### Response: (model should continue)."""
    return f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
You are an AI sales coach. Correct the user gently and concisely.

### Input:
Ground truth: {ground_truth}
User said: {user_utterance}

### Response:
"""


def split_prompt_fields(prompt_block):
    """Pull ground-truth / user-said lines out of our stored prompt string."""
    # prompt_block looks like:
    #   "Ground truth: ...\nUser said: ..."
    lines = prompt_block.split("\n")
    truth = lines[0].replace("Ground truth: ", "", 1)
    user = lines[1].replace("User said: ", "", 1)
    return truth, user


# Build full documents = formatted prompt + completion (chosen AND rejected)
# so BPE sees every word that will appear in training.
all_texts = []
for ex in preference_data:
    truth, user = split_prompt_fields(ex["prompt"])
    head = format_prompt(truth, user)
    all_texts.append(head + ex["chosen"])
    all_texts.append(head + ex["rejected"])

# ---------------------------------------------------------------------------
# Tokenizer: REUSE SFT vocab (do NOT train a fresh BPE here)
# ---------------------------------------------------------------------------
# Sticky cause of the 260 vs 277 error:
#   correction_gpt_sft.pt was trained with SFT BPE vocab size 260.
#   Training a NEW BPE on preference text made vocab 277 → embed/head
#   shapes no longer match the checkpoint.
#
# Production / this lesson: load the tokenizer file saved next to the .pt
# (notebook 9 writes correction_gpt_sft_tokenizer.json).

_tok_candidates = [
    Path("correction_gpt_sft_tokenizer.json"),
    Path("week2") / "correction_gpt_sft_tokenizer.json",
    Path.cwd() / "correction_gpt_sft_tokenizer.json",
    Path.cwd() / "week2" / "correction_gpt_sft_tokenizer.json",
]
_tok_path = next((p for p in _tok_candidates if p.is_file()), None)

tokenizer = BPETokenizer()
if _tok_path is not None:
    tokenizer.load(_tok_path)
else:
    # Fallback only for demos with no SFT artifacts: train on preference text
    # (then you CANNOT load correction_gpt_sft.pt — shapes will disagree).
    print("No SFT tokenizer file found — training toy BPE on preference text.")
    print("Save tokenizer from notebook 9 to load SFT weights in the next cell.")
    tokenizer = BPETokenizer(vocab_size=500)
    tokenizer.train("\n".join(all_texts))

print(f"preference documents built: {len(all_texts)}  (3 pairs × 2 completions)")
print(f"tokenizer vocab_size: {len(tokenizer.vocab)}")

# ---------------------------------------------------------------------------
# HOW TO READ THE OUTPUT
# ---------------------------------------------------------------------------
# preference pairs: 3 / Sample pair: chosen = short Correction, rejected = hedgy.
#
# Loaded tokenizer ← .../correction_gpt_sft_tokenizer.json (vocab=260)
#   SUCCESS path — same jersey numbers as correction_gpt_sft.pt.
#   Next cell should print "Loaded SFT weights".
#
# If you still see vocab=277 / merge logs:
#   Fresh BPE fallback ran (no tokenizer JSON). Re-run SFT save cell, or
#   keep the week2/correction_gpt_sft_tokenizer.json file next to the .pt.
#
# Next: load MiniGPT policy + frozen ref with matching vocab → DPO loop.


preference pairs: 3
Sample pair:
 {
  "prompt": "Ground truth: The price is $500/month\nUser said: It costs $300 a month",
  "chosen": "Correction: The price is $500 per month, not $300.",
  "rejected": "Well, you may want to double-check that because I think the actual price is somewhere around $500, but I'm not entirely sure, let me look it up..."
}
Loaded tokenizer ← correction_gpt_sft_tokenizer.json (vocab=260)
preference documents built: 6  (3 pairs × 2 completions)
tokenizer vocab_size: 260


In [5]:
# =============================================================================
# DPO LOSS — how we score "chosen better than rejected"
# =============================================================================
#
# ---------------------------------------------------------------------------
# TOPIC: two models, four numbers, one push
# ---------------------------------------------------------------------------
# POLICY model  = the student we TRAIN (weights move)
# REFERENCE     = a FROZEN copy (usually SFT) — the "don't forget who you were"
#                 baseline. We compare against it so the student doesn't go wild.
#
# For EACH preference pair we need FOUR scores:
#
#   policy_chosen_logprob   how much the STUDENT likes the good answer
#   policy_rejected_logprob how much the STUDENT likes the bad answer
#   ref_chosen_logprob      how much the FROZEN SFT liked the good answer
#   ref_rejected_logprob    how much the FROZEN SFT liked the bad answer
#
# Easy story:
#   "Become MORE sure that chosen > rejected than the old SFT already was."
#
#
# ---------------------------------------------------------------------------
# PICTURE (one training example)
# ---------------------------------------------------------------------------
#
#   PROMPT ─────────────────────────────────────┐
#                                               │
#                    ┌──────────────────────────┼──────────────────────────┐
#                    ▼                          │                          ▼
#              CHOSEN ✅                   same prompt              REJECTED ❌
#         short Correction: ...                                 hedgy essay...
#                    │                                                 │
#                    ▼                                                 ▼
#              logprob under POLICY                              logprob under POLICY
#              logprob under REF                                 logprob under REF
#                    │                                                 │
#                    └────────────┬────────────────────────────────────┘
#                                 ▼
#                    DPO loss: raise chosen relative to rejected
#                    (vs what the reference already believed)
#
#
# ---------------------------------------------------------------------------
# MATH IN PLAIN WORDS (matches the code below)
# ---------------------------------------------------------------------------
#   policy_logratios = policy_chosen - policy_rejected
#       "How much MORE does the student like chosen than rejected?"
#
#   ref_logratios    = ref_chosen - ref_rejected
#       "How much MORE did the frozen SFT like chosen than rejected?"
#
#   logits = policy_logratios - ref_logratios
#       "Did the student IMPROVE that preference gap vs the reference?"
#       Positive logits  → student prefers chosen (vs rejected) MORE than SFT did
#       Negative logits  → student got worse / flipped
#
#   loss = -logsigmoid(beta * logits)
#       Soft "please make logits positive." Bigger beta = stay closer to
#       reference (smaller allowed change). Smaller beta = freer to move.
#
# Sticky: we never train a separate reward model. The preference is baked
# into this one loss. That's the DPO trick vs classic RLHF.
#

import torch.nn.functional as F


def dpo_loss(
    policy_chosen_logprobs,
    policy_rejected_logprobs,
    ref_chosen_logprobs,
    ref_rejected_logprobs,
    beta=0.1,
):
    """DPO loss for a batch of preference pairs (paper-style).

    Each *_logprobs argument is a vector of shape (B,) —
    usually the mean log-prob of response tokens under that model.
    beta: how strongly we penalize drifting from the reference.
    """
    # Gap under the student: chosen should beat rejected
    policy_logratios = policy_chosen_logprobs - policy_rejected_logprobs
    # Same gap under the frozen reference (SFT)
    ref_logratios = ref_chosen_logprobs - ref_rejected_logprobs

    # Improvement of that gap vs reference (the quantity DPO pushes up)
    logits = policy_logratios - ref_logratios

    # -log sigmoid: low loss when logits are large positive
    loss = -F.logsigmoid(beta * logits).mean()
    return loss


# ---------------------------------------------------------------------------
# How to get those log-probs from MiniGPT
# ---------------------------------------------------------------------------
# Feed FULL sequence:  [prompt tokens | response tokens]
# Model at seat t predicts token t+1 (same causal shift as always).
# We ONLY average log-probs on RESPONSE seats — prompt is context, not graded.
#
# Picture of the shift window:
#
#   ids:   [ p0 p1 p2 | r0 r1 r2 r3 ]     response starts at index s
#                    s
#   logits at s-1 predicts r0
#   logits at s   predicts r1
#   ...
#   We gather log P(true token) at each of those seats, then mean.
#

def compute_logprob(model, input_ids, response_start_idx):
    """Average log-probability of response tokens under `model`.

    input_ids:         (B, T) full sequence = prompt + response
    response_start_idx: index of the first response token
    returns:           (B,) mean log-prob over response tokens
    """
    logits = model(input_ids)  # (B, T, vocab)

    # Causal shift: logits[t] predicts token[t+1]
    # Start predicting at response_start_idx → use logits at response_start_idx-1
    shift_logits = logits[:, response_start_idx - 1 : -1, :]
    shift_labels = input_ids[:, response_start_idx:]

    log_probs = F.log_softmax(shift_logits, dim=-1)
    # Pick the log-prob of the ACTUAL next token at each seat
    token_log_probs = torch.gather(
        log_probs, 2, shift_labels.unsqueeze(-1)
    ).squeeze(-1)

    # Toy setup: short responses, no PAD masking yet → simple mean
    return token_log_probs.mean(dim=1)  # (B,)


# ---------------------------------------------------------------------------
# HOW TO READ THIS CELL (no big printout yet)
# ---------------------------------------------------------------------------
# Running it only DEFINES the two helpers. Next cell usually:
#   1) load / clone MiniGPT as policy + frozen reference
#   2) for each preference row, encode prompt+chosen and prompt+rejected
#   3) call compute_logprob four times → dpo_loss(...) → backward on policy
#
# Tiny numeric intuition (single pair, made-up numbers):
#   policy_chosen= -1.0, policy_rejected= -3.0  → policy_logratios = +2.0
#   ref_chosen=    -1.5, ref_rejected=    -2.0  → ref_logratios    = +0.5
#   logits = 2.0 - 0.5 = +1.5  → student already prefers chosen more than SFT
#   loss = -logsigmoid(0.1 * 1.5)  ≈ small  → gentle nudge, already good direction
#
# If logits were negative, loss would be larger → stronger push to fix it.


In [11]:
# =============================================================================
# LOAD POLICY + FROZEN REFERENCE (the two brains DPO needs)
# =============================================================================
#
# ---------------------------------------------------------------------------
# TOPIC: why TWO copies of MiniGPT?
# ---------------------------------------------------------------------------
# POLICY  = student we TRAIN during DPO (weights move)
# REFERENCE = frozen snapshot (usually the SFT checkpoint)
#             answers: "how much did the OLD model already like chosen vs rejected?"
#
# Picture:
#   ┌─────────────────┐         ┌─────────────────┐
#   │  policy_model   │ train   │   ref_model     │ frozen
#   │  (student)      │ ◀────── │   (SFT twin)    │ no grad
#   └────────┬────────┘         └────────┬────────┘
#            │                           │
#            └──────────┬────────────────┘
#                       ▼
#              dpo_loss( four logprobs )
#
# Sticky: reference does NOT learn. If both moved, the "baseline" would
# chase the student and the preference signal would collapse.
#
# Shared module: week2/mini_gpt.py  (NOT day8_minigpt — that file does not exist)
# Cell 0 already put week2/ on sys.path.
#

from pathlib import Path
from mini_gpt import MiniGPT

vocab_size = len(tokenizer.vocab)
# Match the SFT checkpoint architecture from notebook 9 when present:
# that save used block_size=64 and vocab≈260. Our DPO BPE may differ
# (~277) — then load will fail and we demo with fresh weights (still OK to learn wiring).
block_size = 64

policy_model = MiniGPT(
    vocab_size,
    embed_dim=64,
    num_heads=4,
    ff_dim=128,
    num_layers=3,
    block_size=block_size,
)

# Look for notebook-9 save in common places
_ckpt_candidates = [
    Path("correction_gpt_sft.pt"),
    Path("week2") / "correction_gpt_sft.pt",
    Path.cwd() / "correction_gpt_sft.pt",
    Path.cwd() / "week2" / "correction_gpt_sft.pt",
]
_ckpt = next((p for p in _ckpt_candidates if p.is_file()), None)

if _ckpt is not None:
    try:
        state = torch.load(_ckpt, map_location="cpu", weights_only=True)
        policy_model.load_state_dict(state)
        print(f"Loaded SFT weights from {_ckpt}")
    except Exception as e:
        # Typical cause: DPO tokenizer vocab_size ≠ SFT checkpoint vocab
        print(f"Could not load {_ckpt}: {e}")
        print("Starting fresh weights (demo). Production DPO reuses the SFT tokenizer.")
else:
    print("No correction_gpt_sft.pt found — starting fresh (for demo).")

# Reference = exact clone of current policy, then FREEZE
ref_model = MiniGPT(
    vocab_size,
    embed_dim=64,
    num_heads=4,
    ff_dim=128,
    num_layers=3,
    block_size=block_size,
)
ref_model.load_state_dict(policy_model.state_dict())
ref_model.eval()
for param in ref_model.parameters():
    param.requires_grad = False  # never update reference

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
policy_model.to(device)
ref_model.to(device)

print(f"device={device} | vocab={vocab_size} | block_size={block_size}")
print(
    f"policy params={sum(p.numel() for p in policy_model.parameters()):,} | "
    f"ref trainable={sum(p.requires_grad for p in ref_model.parameters())}"
)
# Expect: ref trainable=0. Next: encode preference pairs → dpo_loss → step policy only.
#
# ---------------------------------------------------------------------------
# HOW TO READ THE OUTPUT
# ---------------------------------------------------------------------------
# "Loaded SFT weights from ..."
#   Great — policy/ref start from notebook-9 SFT (true preference fine-tune).
#
# "Could not load ... size mismatch ..."
#   Cell 0 used a DIFFERENT vocab than the .pt (old bug: fresh BPE → 277).
#   Fix: load correction_gpt_sft_tokenizer.json in cell 0 (vocab 260), re-run.
#
# "No correction_gpt_sft.pt found"
#   Run / save SFT notebook first, or keep demo-from-scratch.
#
# ref trainable=0  → freeze worked. Only policy_model will get optimizer steps.


Loaded SFT weights from correction_gpt_sft.pt
device=cpu | vocab=260 | block_size=64
policy params=137,604 | ref trainable=0
